# ERA5 Pipeline — Annotated Walkthrough

A step-by-step learning reference for the ERA5 data pipeline.
Covers real ARCO ERA5 from Google Cloud → local zarr → regrid → Dask stats → PyTorch DataLoader.

**Variables used:**
- `2m_temperature` — surface air temperature
- `volumetric_soil_water_layer_1` — top-layer soil moisture
- `leaf_area_index_high_vegetation` — vegetation state

**Source:** `gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3`

In [ ]:
from __future__ import annotations
import os, shutil, time, warnings
from typing import Sequence

import numpy as np
import torch
import xarray as xr
import dask
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore', category=UserWarning)

## Config

Keep the first pass small — 2 weeks over CONUS fits comfortably in local dev memory.

In [ ]:
ARCO_ERA5_PATH = 'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3'

VARIABLES = [
    '2m_temperature',
    'volumetric_soil_water_layer_1',
    'leaf_area_index_high_vegetation',
]

TIME_START = '2020-06-01'
TIME_END   = '2020-06-14'   # 2 weeks = 336 hourly timesteps

# CONUS bounding box (ERA5 uses 0–360 longitude)
LAT_MAX =  50.0
LAT_MIN =  25.0
LON_MIN = 235.0   # ~125°W
LON_MAX = 295.0   # ~65°W

LOCAL_ZARR_PATH   = '../data/era5_subset.zarr'
LOCAL_REGRID_PATH = '../data/era5_subset_1deg.zarr'

os.makedirs('../data', exist_ok=True)

## Dask Setup

On M1/8GB, the **threaded scheduler** outperforms `LocalCluster`:
- No subprocess spawning (avoids `__main__` guard issues)
- No port conflicts
- Lower memory overhead
- Threads release the GIL for numpy/zarr I/O so parallelism still works

`'synchronous'` = single thread (good for debugging)  
`'threads'` = thread pool (good for I/O-heavy workloads like zarr reads)

In [ ]:
dask.config.set(scheduler='threads', num_workers=4)
print('Dask scheduler: threaded (4 workers, no LocalCluster)')

## Step 1: Open & Subset ARCO ERA5

`xr.open_zarr` is **lazy** — nothing is downloaded until `.compute()` is called.

Key points:
- `chunks={'time': 1}` tells Dask to treat each timestep as its own task
- ERA5 latitude is stored **descending** (90 → -90), so `slice(lat_max, lat_min)`
- Rename `latitude/longitude` → `lat/lon` for xESMF compatibility
- Derived variables (e.g. `t2m_celsius`) stay lazy — no `.compute()` yet

In [ ]:
print('Opening ARCO ERA5 (lazy — nothing downloaded yet)...')

ds = xr.open_zarr(
    ARCO_ERA5_PATH,
    consolidated=True,
    storage_options={'token': 'anon'},
    chunks={'time': 1},
)

print(f'Remote vars available: {list(ds.data_vars)[:8]} ...')

# Subset: only the variables + region + time window we care about
subset = ds[VARIABLES].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_MAX, LAT_MIN),   # descending → slice(max, min)
    longitude=slice(LON_MIN, LON_MAX),
)

subset = subset.rename({'latitude': 'lat', 'longitude': 'lon'})

# Derived variable — still lazy
subset['t2m_celsius'] = subset['2m_temperature'] - 273.15
subset['t2m_celsius'].attrs = {'long_name': '2m temperature', 'units': 'degC'}

print(f'Subset dims: {dict(subset.sizes)}')

# Rough memory estimate before committing to a .compute()
n_cells = 1
for v in subset.sizes.values():
    n_cells *= v
mb = n_cells * len(subset.data_vars) * 4 / 1e6
print(f'Estimated full load size: {mb:.0f} MB across {len(subset.data_vars)} variables')
if mb > 2000:
    print('  >2GB — consider narrowing TIME_END or bbox')

subset

## Step 2: Write Local Zarr

### Chunking strategy: `{time: 1, lat: -1, lon: -1}`

| Access pattern | Optimal chunks |
|---|---|
| ML training (sequential frames) | `{time: 1, lat: -1, lon: -1}` |
| Time-series analysis | `{time: -1, lat: 10, lon: 10}` |
| Spatial statistics | `{time: 24, lat: -1, lon: -1}` |

With `time: 1`, each chunk is one full spatial snapshot (~0.1 MB for CONUS at 0.25°).
ML DataLoader loads frames sequentially → one chunk per `__getitem__` call.

**Important:** `drop_encoding()` clears inherited remote encodings from ARCO ERA5.
Without this, zarr v3 codec format mismatches cause write errors.

In [ ]:
ML_CHUNKS = {'time': 1, 'lat': -1, 'lon': -1}  # -1 = full dimension

# Clear inherited remote encodings — required to avoid zarr v3 codec conflicts
subset_chunked = subset.drop_encoding().chunk(ML_CHUNKS)
for v in list(subset_chunked.data_vars) + list(subset_chunked.coords):
    subset_chunked[v].encoding = {}

if os.path.exists(LOCAL_ZARR_PATH):
    shutil.rmtree(LOCAL_ZARR_PATH)

print(f'Chunk layout: {ML_CHUNKS}')
print(f'Writing to {LOCAL_ZARR_PATH} ...')

t0 = time.time()
subset_chunked.to_zarr(
    LOCAL_ZARR_PATH,
    mode='w',
    consolidated=True,   # writes consolidated metadata — faster remote reads
)
print(f'Done in {time.time() - t0:.1f}s')

# Re-open from local — now fully offline
ds_local = xr.open_zarr(LOCAL_ZARR_PATH, consolidated=True, chunks=ML_CHUNKS)
print(f'Local store dims: {dict(ds_local.sizes)}')
ds_local

## Step 3: Regrid with xESMF

### Which method to use?

| Method | Use for | Preserves |
|---|---|---|
| `bilinear` | Temperature, wind, geopotential, soil moisture | Local point values |
| `conservative` | Precipitation, radiation, surface fluxes | Area integrals (mass/energy) |
| `nearest_s2d` | Categorical / mask fields | Discrete categories |

Conservative regridding is physically critical: bilinear-interpolated precipitation
can violate conservation of mass. ERA5 0.25° → WRF boundary conditions always
requires conservative for flux variables.

In [ ]:
try:
    import xesmf as xe

    target_grid = xr.Dataset({
        'lat': (['lat'], np.arange(LAT_MIN, LAT_MAX + 1.0, 1.0)),
        'lon': (['lon'], np.arange(LON_MIN, LON_MAX + 1.0, 1.0)),
    })

    print(f'Source: {ds_local.sizes["lat"]} × {ds_local.sizes["lon"]} (0.25°)')
    print(f'Target: {len(target_grid.lat)} × {len(target_grid.lon)} (1.0°)')

    state_vars = ['2m_temperature', 'volumetric_soil_water_layer_1',
                  'leaf_area_index_high_vegetation', 't2m_celsius']
    state_vars = [v for v in state_vars if v in ds_local]

    regridder = xe.Regridder(
        ds_local[state_vars],
        target_grid,
        method='bilinear',
        periodic=False,
        reuse_weights=True,   # cache weights — saves time if called repeatedly
    )

    ds_regrid = regridder(ds_local[state_vars])
    print(f'Regridded dims: {dict(ds_regrid.sizes)}')

    if os.path.exists(LOCAL_REGRID_PATH):
        shutil.rmtree(LOCAL_REGRID_PATH)

    ds_regrid.chunk({'time': 1, 'lat': -1, 'lon': -1}).to_zarr(
        LOCAL_REGRID_PATH, mode='w', consolidated=True
    )
    print(f'Saved: {LOCAL_REGRID_PATH}')

except ImportError:
    print('xESMF not installed — skipping regrid step.')
    print('Install: conda install -c conda-forge xesmf')

## Step 4: Dask Parallel Stats

### Key concept: lazy evaluation

```python
# Without Dask:
ds = xr.open_dataset('huge_era5.nc')  # loads everything into RAM → crash
mean = ds.t850.mean()                 # computes on full array

# With Dask:
ds = xr.open_zarr('era5.zarr')        # loads nothing — just metadata
mean = ds.t850.mean()                 # builds a task graph — nothing computed yet
result = mean.compute()              # NOW it executes, chunk by chunk
```

Build **all** graphs before calling `.compute()` — Dask can then fuse operations
and avoid reading each chunk more than once.

In [ ]:
# Build all lazy graphs before computing any
temp_mean = (ds_local['2m_temperature'] - 273.15).mean('time')
soil_std  = ds_local['volumetric_soil_water_layer_1'].std('time')
lai_max   = ds_local['leaf_area_index_high_vegetation'].max('time')

print('Lazy graphs built — computing now ...')
t0 = time.time()

# Compute all three in one pass — Dask reads each chunk once for all ops
temp_mean_val, soil_std_val, lai_max_val = dask.compute(
    temp_mean, soil_std, lai_max
)
elapsed = time.time() - t0

print(f'Computed in {elapsed:.2f}s')
print(f'Mean 2m temp (°C):          {float(temp_mean_val.mean()):.2f}')
print(f'Soil moisture temporal std:  {float(soil_std_val.mean()):.4f}')
print(f'LAI high veg max:            {float(lai_max_val.mean()):.4f}')

## Step 5: PyTorch Dataset / DataLoader

### Design decisions

- `__getitem__` uses `isel(time=slice(...))` — loads only `seq_len+1` chunks, not the full dataset
- Normalization stats computed once at init, stored as scalars. In production: compute over a full year and save to JSON
- `num_workers=0`: on M1, multiprocessing with fork can OOM due to copy-on-write memory. Use 0 for local dev
- `pin_memory=False`: only helps with CUDA, not MPS

In [ ]:
class ERA5Dataset(Dataset):
    """
    Zarr-backed sliding window dataset for ConvLSTM training.

    Returns:
        x: (seq_len, C, H, W)  — input sequence
        y: (C, H, W)           — target next frame
    """

    def __init__(self, zarr_path: str, variables: Sequence[str], seq_len: int = 6) -> None:
        self.ds        = xr.open_zarr(zarr_path, consolidated=True)[list(variables)]
        self.variables = list(variables)
        self.seq_len   = seq_len
        self.n_times   = self.ds.sizes['time']

        print('  Computing normalization stats ...')
        self.means: dict[str, float] = {}
        self.stds:  dict[str, float] = {}

        for v in self.variables:
            mean_val = float(self.ds[v].mean().compute())
            std_val  = float(self.ds[v].std().compute())
            self.means[v] = mean_val
            self.stds[v]  = std_val if std_val > 0 else 1.0
            print(f'    {v}: mean={mean_val:.4f}  std={std_val:.4f}')

    def __len__(self) -> int:
        return self.n_times - self.seq_len

    def __getitem__(self, idx: int):
        # isel with slice loads only these chunks — not the full dataset
        window = self.ds.isel(time=slice(idx, idx + self.seq_len + 1))

        arr = np.stack(
            [((window[v].values - self.means[v]) / self.stds[v]).astype(np.float32)
             for v in self.variables],
            axis=1,
        )  # (T+1, C, H, W)

        x = torch.from_numpy(arr[:self.seq_len])   # (seq_len, C, H, W)
        y = torch.from_numpy(arr[self.seq_len])    # (C, H, W)
        return x, y

In [ ]:
dataset = ERA5Dataset(
    zarr_path=LOCAL_ZARR_PATH,
    variables=['2m_temperature', 'volumetric_soil_water_layer_1', 'leaf_area_index_high_vegetation'],
    seq_len=6,
)

print(f'Dataset length: {len(dataset)} windows')
x, y = dataset[0]
print(f'Sample x: {x.shape}  (seq_len=6, C=3, H, W)')
print(f'Sample y: {y.shape}  (C=3, H, W)')
print(f'x mean: {x.mean():.4f}  std: {x.std():.4f}  (should be ~0, ~1)')

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=False)
batch_x, batch_y = next(iter(loader))
print(f'Batch x: {batch_x.shape}  (B=4, T=6, C=3, H, W)')
print(f'Batch y: {batch_y.shape}  (B=4, C=3, H, W)')

## Cloud Storage Patterns

The zarr API is identical for local and cloud stores — just swap the path:

```python
# Read public ARCO ERA5 (no credentials)
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    consolidated=True,
    storage_options={'token': 'anon'},
    chunks={'time': 1},
)

# Write to GCS
ds.to_zarr('gs://your-bucket/era5_processed.zarr', mode='w', consolidated=True)

# Write to S3
import s3fs
store = s3fs.S3Map('s3://your-bucket/era5_processed.zarr', s3=s3fs.S3FileSystem())
ds.to_zarr(store, mode='w', consolidated=True)
```

Point `ERA5Dataset` at a cloud zarr to train directly from cloud storage:

```python
dataset = ERA5Dataset(
    zarr_path='gs://your-bucket/era5_processed.zarr',
    variables=['2m_temperature', ...],
)
```